# Cumulocity → Snowflake: tear down

**Irreversible.** Removes everything `cumulocity_build_up.ipynb` created for the tenant set below, in reverse dependency order. Other tenants are not touched. Every statement uses `IF EXISTS`, so the notebook can be re-run after a partial build or failure.

| Step | Removes |
|---|---|
| 1. Schedules | Task and alert are stopped first, so nothing refreshes or sends e-mail during the drop |
| 2. App database | `CUMULOCITY_<T>`: dynamic tables, procedures, `SCHEMA_STATE`, task, alert |
| 3. Iceberg connection | Raw database `CUMULOCITY_ICEBERG_<T>`, catalog integration `POLARIS_CUMULOCITY_INTEGRATION_<T>` |
| 4. Account objects | E-mail integration `CUMULOCITY_ALERT_EMAIL_<T>`, warehouse `CUMULOCITY_WH_<T>` |
| 5. Check | Confirms nothing is left |

The Iceberg data in Cumulocity's storage is never touched; dropping the raw database only removes the Snowflake-side link.

## Parameters

Set `catalog_name` to the tenant to remove. The names are derived exactly as in the build-up notebook.

In [ ]:
import re

catalog_name = ""   # Cumulocity tenant ID

assert re.fullmatch(r"[A-Za-z0-9]+", catalog_name), "catalog_name must be alphanumeric (used in object names)"
tenant      = catalog_name.upper()
integration = f"POLARIS_CUMULOCITY_INTEGRATION_{tenant}"
raw_db      = f"CUMULOCITY_ICEBERG_{tenant}"
app_db      = f"CUMULOCITY_{tenant}"
wh          = f"CUMULOCITY_WH_{tenant}"
email_int   = f"CUMULOCITY_ALERT_EMAIL_{tenant}"

print("Will remove:")
for value in [app_db, raw_db, integration, email_int, wh]:
    print("  ", value)

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

## 1. Schedules

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# IF EXISTS doesn't help once the database itself is gone, so skip instead of failing on a re-run
for stmt in [f"ALTER TASK IF EXISTS {app_db}.OPS.REBUILD_ON_SCHEMA_CHANGE SUSPEND",
             f"ALTER ALERT IF EXISTS {app_db}.OPS.DT_REFRESH_FAILURE_ALERT SUSPEND"]:
    try:
        session.sql(stmt).collect()
        print("done:   ", stmt)
    except Exception as e:
        if "does not exist" not in str(e):
            raise
        print("skipped:", stmt, "(database not found)")

## 2. App database

Removes `CURATED` (dynamic tables) and `OPS` (procedures, fingerprint table, task, alert) in one step.

In [ ]:
%%sql -r drop_app_db
DROP DATABASE IF EXISTS {{app_db}}

## 3. Iceberg connection

The raw database depends on the integration, so it goes first.

In [ ]:
%%sql -r drop_raw_db
DROP DATABASE IF EXISTS {{raw_db}}

In [ ]:
%%sql -r drop_integration
DROP CATALOG INTEGRATION IF EXISTS {{integration}}

## 4. Account objects

In [ ]:
%%sql -r drop_email_integration
DROP NOTIFICATION INTEGRATION IF EXISTS {{email_int}}

In [ ]:
%%sql -r drop_warehouse
DROP WAREHOUSE IF EXISTS {{wh}}

## 5. Check

Each cell should return no rows.

In [ ]:
%%sql -r remaining_integrations
SHOW INTEGRATIONS LIKE '%{{tenant}}'

In [ ]:
%%sql -r remaining_databases
SHOW DATABASES LIKE 'CUMULOCITY%{{tenant}}'

In [ ]:
%%sql -r remaining_warehouse
SHOW WAREHOUSES LIKE '{{wh}}'